# JobHop v2 LightOn LateOn retrieval evaluation

This notebook evaluates the LightOn late-interaction model requested by the supervisor.

Model:

- `lightonai/LateOn`

Why separate from notebook 09:

- BGE-M3, Qwen3-Embedding, and Nemotron3-Embed are dense single-vector embedding models.
- LateOn is a ColBERT-style multi-vector / late-interaction model.
- The scoring pipeline is different, so it belongs in a separate notebook.

Task/protocol:

- public JobHop v2;
- ESCO metadata title/description;
- last-step next-occupation prediction;
- MRR / Recall@5 / Recall@10.

The default run is small (`MAX_EVAL_CASES = 2000`) so it matches the sampled dense-model screening in notebook 09, while still staying manageable for late-interaction retrieval.

In [1]:
# One-click Colab setup.
!pip -q install -U pylate sentence-transformers datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.6/185.6 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.4/512.4 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 422.8/422.8 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 112.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/

In [2]:
import gc
import os
import random
import re
import shutil
import traceback
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import requests
import torch
from datasets import load_dataset
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 180)

## Configuration

Start small. If this runs successfully:

- increase `MAX_EVAL_CASES` to 2000 or 5000;
- increase `LATEON_RETRIEVE_K` to 300 or 500;
- only then consider full test.

This notebook evaluates two things:

1. pure LateOn retrieval over all 2,982 candidate occupations;
2. a small hybrid reranking over the union of LateOn top-k and Markov top-k candidates.

In [3]:
DATASET_ID = "aida-ugent/JobHop"
HF_API = f"https://huggingface.co/api/datasets/{DATASET_ID}/tree/main"
SPLITS = ("train", "validation", "test")

MAX_EVAL_CASES = 2000
RANDOM_SEED = 7

LATEON_MODEL_NAME = "lightonai/LateOn"
LATEON_RETRIEVE_K = 200
MARKOV_CANDIDATE_K = 200
ALPHAS = [0.0, 0.25, 0.5, 0.75, 0.9, 1.0]

REMOVE_UNKNOWN = True
COLLAPSE_CONSECUTIVE_DUPLICATES = True

COLBERT_BATCH_SIZE_DOCS = 32
COLBERT_BATCH_SIZE_QUERIES = 16
INDEX_FOLDER = "pylate-jobhop-index"
INDEX_NAME = "esco_occupations"

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

## Load public JobHop v2

In [4]:
def hf_tree(path=""):
    url = HF_API if not path else f"{HF_API}/{path}"
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    return response.json()


def list_parquet_files():
    files = []
    stack = [""]
    while stack:
        path = stack.pop()
        try:
            entries = hf_tree(path)
        except Exception:
            continue
        for item in entries:
            item_path = item.get("path") or item.get("name")
            if not item_path:
                continue
            if item_path.endswith(".parquet"):
                files.append(item_path)
            elif item.get("type") == "directory":
                stack.append(item_path)
    return sorted(set(files))


def split_from_path(path):
    lower = path.lower()
    for split in SPLITS:
        if re.search(rf"(^|/|_|-)({split})(/|_|-|\.|$)", lower):
            return split
    if "val" in lower:
        return "validation"
    return None


parquet_files = list_parquet_files()
split_files = {split: [] for split in SPLITS}
for path in parquet_files:
    split = split_from_path(path)
    if split in split_files:
        split_files[split].append(path)

print(parquet_files)
split_files

['JobHop_v2_test.parquet', 'JobHop_v2_train.parquet', 'JobHop_v2_val.parquet']


{'train': ['JobHop_v2_train.parquet'],
 'validation': ['JobHop_v2_val.parquet'],
 'test': ['JobHop_v2_test.parquet']}

In [5]:
def hf_resolve_url(path):
    return f"https://huggingface.co/datasets/{DATASET_ID}/resolve/main/{path}"


def load_split(split):
    paths = split_files.get(split, [])
    if not paths:
        raise RuntimeError(f"No parquet files found for split={split}. Files seen: {parquet_files[:20]}")
    frames = []
    for path in tqdm(paths, desc=f"loading {split}"):
        frames.append(pd.read_parquet(hf_resolve_url(path)))
    df = pd.concat(frames, ignore_index=True)
    df["split"] = split
    return df


raw = {split: load_split(split) for split in SPLITS}
for split, df in raw.items():
    print(split, df.shape, df.columns.tolist())

loading train:   0%|          | 0/1 [00:00<?, ?it/s]

loading validation:   0%|          | 0/1 [00:00<?, ?it/s]

loading test:   0%|          | 0/1 [00:00<?, ?it/s]

train (1594827, 6) ['resume_id', 'matched_code', 'start_date', 'end_date', 'university_level', 'split']
validation (199587, 6) ['resume_id', 'matched_code', 'start_date', 'end_date', 'university_level', 'split']
test (198877, 6) ['resume_id', 'matched_code', 'start_date', 'end_date', 'university_level', 'split']


In [6]:
def standardize_jobhop_schema(df):
    df = df.copy()
    if "person_id" not in df.columns:
        if "resume_id" in df.columns:
            df["person_id"] = df["resume_id"]
        else:
            raise KeyError(f"No person/resume identifier column found. Columns: {df.columns.tolist()}")

    if "university_studies" not in df.columns:
        if "university_level" in df.columns:
            df["university_studies"] = df["university_level"]
        else:
            df["university_studies"] = pd.NA

    df["matched_code"] = df["matched_code"].astype(str).str.strip()
    return df


raw = {split: standardize_jobhop_schema(df) for split, df in raw.items()}
pd.DataFrame([
    {
        "split": split,
        "rows": len(df),
        "resumes": df["person_id"].nunique(),
        "unknown_code_rows": (df["matched_code"].str.lower() == "unknown").sum(),
    }
    for split, df in raw.items()
])

,split,rows,resumes,unknown_code_rows
0,train,1594827,284251,110573
1,validation,199587,35531,13622
2,test,198877,35533,13587


## Load ESCO metadata

In [7]:
def clean_text(value):
    if value is None or pd.isna(value):
        return ""
    return str(value).replace("\n", " ").strip()


def normalize_esco_code(value):
    if value is None or pd.isna(value):
        return ""
    return str(value).strip().rstrip("/").split("/")[-1]


def add_meta(lookup, code, label="", description="", source=""):
    code = normalize_esco_code(code)
    if not code:
        return
    label = clean_text(label)
    description = clean_text(description)
    if code not in lookup or (not lookup[code].get("description") and description):
        lookup[code] = {
            "label": label or lookup.get(code, {}).get("label", ""),
            "description": description or lookup.get(code, {}).get("description", ""),
            "source": source,
        }


def load_danieldux_esco_metadata():
    lookup = {}
    frames = []
    parquet_urls = [
        "https://huggingface.co/datasets/danieldux/ESCO/resolve/main/data/en/train.parquet",
        "https://huggingface.co/datasets/danieldux/ESCO/resolve/main/data/en/test.parquet",
    ]
    for url in parquet_urls:
        try:
            frames.append(pd.read_parquet(url))
            print("Loaded danieldux parquet:", url)
        except Exception as exc:
            print("Could not read danieldux parquet:", url, exc)

    if not frames:
        ds = load_dataset("danieldux/ESCO", split="train")
        frames.append(ds.to_pandas())

    df = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["ESCO_CODE"])
    print("Loaded danieldux/ESCO:", df.shape, df.columns.tolist())
    for _, row in df.iterrows():
        add_meta(
            lookup,
            row.get("ESCO_CODE"),
            row.get("ESCO_OCCUPATION", ""),
            row.get("ESCO_DESCRIPTION", ""),
            "hf_danieldux_esco",
        )
    print("danieldux lookup entries:", len(lookup))
    return lookup


metadata_lookup = load_danieldux_esco_metadata()
list(metadata_lookup.items())[:3]

Loaded danieldux parquet: https://huggingface.co/datasets/danieldux/ESCO/resolve/main/data/en/train.parquet
Loaded danieldux parquet: https://huggingface.co/datasets/danieldux/ESCO/resolve/main/data/en/test.parquet
Loaded danieldux/ESCO: (3008, 19) ['ISCO_CODE_1', 'ISCO_LABEL_1', 'ISCO_DEFINITION_1', 'ISCO_CODE_2', 'ISCO_LABEL_2', 'ISCO_DEFINITION_2', 'ISCO_CODE_3', 'ISCO_LABEL_3', 'ISCO_DEFINITION_3', 'ISCO_CODE_4', 'ISCO_LABEL_4', 'ISCO_DEFINITION_4', 'ISCO_CODES', 'ISCO_LABELS', 'ESCO_CODE', 'ESCO_LABELS', 'ESCO_OCCUPATION', 'ESCO_DESCRIPTION', 'LANGUAGE']
danieldux lookup entries: 3007


[('1223.1',
  {'label': 'product manager',
   'description': 'Product managers are responsible for managing the lifecycle of a product. They research and develop new products in addition to managing existing ones through market research and strategic planning. Product managers perform marketing and planning activities to increase profits.',
   'source': 'hf_danieldux_esco'}),
 ('3324.4.8',
  {'label': 'wholesale merchant in computers, computer peripheral equipment and software',
   'description': 'Wholesale merchants in computers, computer peripheral equipment and software investigate potential wholesale buyers and suppliers and match their needs. They conclude trades involving large quantities of goods.',
   'source': 'hf_danieldux_esco'}),
 ('7543.9.3',
  {'label': 'metal product quality control inspector',
   'description': 'Metal product quality control inspectors perform preventive and operational quality control on the metal products. They examine the materials at various stages 

## Build trajectories and cases

In [8]:
def date_key(value):
    if pd.isna(value):
        return (9999, 9)
    text = str(value).strip()
    if not text or text in {"-", "nan", "None"}:
        return (9999, 9)
    m = re.search(r"(19|20)\d{2}", text)
    year = int(m.group(0)) if m else 9999
    q = 9
    qm = re.search(r"[Qq][-_ ]?([1-4])", text)
    if qm:
        q = int(qm.group(1))
    return (year, q)


def normalize_jobhop_code(value):
    if value is None or pd.isna(value):
        return None
    text = str(value).strip()
    if not text or text.lower() in {"nan", "none"}:
        return None
    if REMOVE_UNKNOWN and text.lower() == "unknown":
        return None
    return text


def compact_consecutive(seq):
    out = []
    for item in seq:
        if not out or item["code"] != out[-1]["code"]:
            out.append(item)
    return out


def make_trajectories(df):
    trajectories = {}
    for person_id, group in tqdm(df.groupby("person_id", sort=False), desc="trajectories"):
        group = group.copy()
        group["_date_key"] = group["start_date"].apply(date_key)
        group = group.sort_values(["_date_key", "end_date", "matched_code"], kind="mergesort")
        seq = []
        for _, row in group.iterrows():
            code = normalize_jobhop_code(row["matched_code"])
            if code is None:
                continue
            seq.append({
                "code": code,
                "education": row.get("university_studies"),
            })
        if COLLAPSE_CONSECUTIVE_DUPLICATES:
            seq = compact_consecutive(seq)
        if len(seq) >= 2:
            trajectories[person_id] = seq
    return trajectories


trajectories = {split: make_trajectories(raw[split]) for split in SPLITS}
pd.DataFrame([
    {"split": split, "trajectories_len_ge_2": len(traj)}
    for split, traj in trajectories.items()
])

trajectories:   0%|          | 0/284251 [00:00<?, ?it/s]

trajectories:   0%|          | 0/35531 [00:00<?, ?it/s]

trajectories:   0%|          | 0/35533 [00:00<?, ?it/s]

,split,trajectories_len_ge_2
0,train,256593
1,validation,32123
2,test,32089


In [9]:
def make_last_step_cases(trajectories_dict):
    cases = []
    for person_id, seq in trajectories_dict.items():
        cases.append({
            "person_id": person_id,
            "prefix": [item["code"] for item in seq[:-1]],
            "target": seq[-1]["code"],
            "education": seq[0].get("education"),
            "prefix_length": len(seq) - 1,
        })
    return cases


validation_cases = make_last_step_cases(trajectories["validation"])
test_cases = make_last_step_cases(trajectories["test"])

candidate_codes = sorted({item["code"] for seq in trajectories["train"].values() for item in seq})
candidate_set = set(candidate_codes)

validation_cases = [case for case in validation_cases if case["target"] in candidate_set]
test_cases = [case for case in test_cases if case["target"] in candidate_set]

if MAX_EVAL_CASES is not None:
    rng = random.Random(RANDOM_SEED)
    validation_cases = rng.sample(validation_cases, min(MAX_EVAL_CASES, len(validation_cases)))
    test_cases = rng.sample(test_cases, min(MAX_EVAL_CASES, len(test_cases)))

metadata_coverage = sum(1 for c in candidate_codes if c in metadata_lookup)
description_coverage = sum(1 for c in candidate_codes if metadata_lookup.get(c, {}).get("description"))

print("candidate occupations:", len(candidate_codes))
print("candidate metadata coverage:", metadata_coverage, f"({metadata_coverage / len(candidate_codes):.1%})")
print("candidate description coverage:", description_coverage, f"({description_coverage / len(candidate_codes):.1%})")
print("validation eval cases:", len(validation_cases))
print("test eval cases:", len(test_cases))

candidate occupations: 2982
candidate metadata coverage: 2982 (100.0%)
candidate description coverage: 2982 (100.0%)
validation eval cases: 2000
test eval cases: 2000


## Text construction

In [10]:
def label_for_code(code):
    meta = metadata_lookup.get(code, {})
    return meta.get("label") or f"ESCO code {code}"


def description_for_code(code):
    return metadata_lookup.get(code, {}).get("description", "")


def compact_desc(text, max_chars=500):
    text = clean_text(text)
    if len(text) > max_chars:
        return text[:max_chars].rsplit(" ", 1)[0] + "..."
    return text


def candidate_text(code):
    label = label_for_code(code)
    desc = compact_desc(description_for_code(code))
    pieces = [f"ESCO occupation code: {code}.", f"Occupation title: {label}."]
    if desc:
        pieces.append(f"Description: {desc}.")
    return " ".join(pieces)


def query_text(case):
    labels = [label_for_code(code) for code in case["prefix"]]
    history = " -> ".join(labels)
    codes = " -> ".join(case["prefix"])
    edu = case.get("education")
    edu_text = "" if pd.isna(edu) else f" Education level: {edu}."
    return (
        f"Career history occupations: {history}. "
        f"Career history ESCO codes: {codes}."
        f"{edu_text} Predict the next ESCO occupation."
    )


candidate_texts = [candidate_text(code) for code in candidate_codes]
validation_query_texts = [query_text(case) for case in validation_cases]
test_query_texts = [query_text(case) for case in test_cases]

print(candidate_texts[0])
print(test_query_texts[0])

ESCO occupation code: 0110.1. Occupation title: air force officer. Description: Air force officers specialise in flying or ground duties, and supervise a team of air force personnel. They coordinate their team's training and welfare, and perform duties specific to their area of specialisation..
Career history occupations: waiter/waitress. Career history ESCO codes: 5131.2. Education level: Secondary school. Predict the next ESCO occupation.


## Markov candidate scores

In [11]:
def train_markov(trajectories_dict):
    transitions = defaultdict(Counter)
    global_next = Counter()
    for seq in trajectories_dict.values():
        codes = [item["code"] for item in seq]
        for a, b in zip(codes[:-1], codes[1:]):
            transitions[a][b] += 1
            global_next[b] += 1

    global_rank = [(code, count / sum(global_next.values())) for code, count in global_next.most_common()]
    transition_probs = {}
    for prev, counter in transitions.items():
        total = sum(counter.values())
        transition_probs[prev] = [(code, count / total) for code, count in counter.most_common()]
    return transition_probs, global_rank


transition_probs, global_rank = train_markov(trajectories["train"])


def top_markov_scores(case, k=200):
    last = case["prefix"][-1]
    ranked = transition_probs.get(last, global_rank)
    out = {}
    for code, score in ranked[:k]:
        out[code] = float(score)
    return out

## LightOn LateOn retrieval with PyLate

In [12]:
from pylate import indexes, models, retrieve

# Clear old index in Colab if rerunning.
if os.path.exists(INDEX_FOLDER):
    shutil.rmtree(INDEX_FOLDER)

print("Loading LateOn:", LATEON_MODEL_NAME)
colbert_model = models.ColBERT(model_name_or_path=LATEON_MODEL_NAME)

print("Encoding candidate documents:", len(candidate_texts))
documents_embeddings = colbert_model.encode(
    candidate_texts,
    batch_size=COLBERT_BATCH_SIZE_DOCS,
    is_query=False,
    show_progress_bar=True,
)

index = indexes.PLAID(
    index_folder=INDEX_FOLDER,
    index_name=INDEX_NAME,
    override=True,
)
index.add_documents(
    documents_ids=candidate_codes,
    documents_embeddings=documents_embeddings,
)
retriever = retrieve.ColBERT(index=index)

Loading LateOn: lightonai/LateOn


modules.json:   0%|          | 0.00/422 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/760 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/3.58M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/581 [00:00<?, ?B/s]

Encoding candidate documents: 2982


Encoding documents (bs=32):   0%|          | 0/94 [00:00<?, ?it/s]

In [13]:
def parse_retrieved_row(row):
    parsed = []
    if isinstance(row, dict):
        ids = row.get("documents_ids") or row.get("document_ids") or row.get("ids")
        scores = row.get("scores") or row.get("score")
        if ids is not None:
            if scores is None:
                scores = [1.0] * len(ids)
            return [(str(i), float(s)) for i, s in zip(ids, scores)]
    if isinstance(row, (list, tuple)):
        for item in row:
            if isinstance(item, dict):
                doc_id = item.get("id") or item.get("document_id") or item.get("documents_id")
                score = item.get("score", item.get("similarity", 0.0))
                if doc_id is not None:
                    parsed.append((str(doc_id), float(score)))
            elif isinstance(item, (list, tuple)) and len(item) >= 2:
                parsed.append((str(item[0]), float(item[1])))
    return parsed


def retrieve_lateon(query_texts):
    print("Encoding queries:", len(query_texts))
    queries_embeddings = colbert_model.encode(
        query_texts,
        batch_size=COLBERT_BATCH_SIZE_QUERIES,
        is_query=True,
        show_progress_bar=True,
    )
    raw = retriever.retrieve(
        queries_embeddings=queries_embeddings,
        k=LATEON_RETRIEVE_K,
    )
    parsed = [parse_retrieved_row(row) for row in raw]
    if parsed and not parsed[0]:
        print("Warning: parser did not recognize retrieval output. First raw row:")
        print(raw[0])
    return parsed


validation_lateon = retrieve_lateon(validation_query_texts)
test_lateon = retrieve_lateon(test_query_texts)
validation_lateon[:1]

Encoding queries: 2000


Encoding queries (bs=16):   0%|          | 0/125 [00:00<?, ?it/s]

Encoding queries: 2000


Encoding queries (bs=16):   0%|          | 0/125 [00:00<?, ?it/s]

[[('5223.5', 30.2255859375),
  ('5223.4', 30.20166015625),
  ('3315.3', 30.1533203125),
  ('5223.6', 30.14599609375),
  ('5223.7', 30.140625),
  ('1346.5', 30.13427734375),
  ('5223.7.25', 30.1337890625),
  ('5223.3', 30.1337890625),
  ('5223.2', 30.13330078125),
  ('5223.7.9', 30.1318359375),
  ('5223.7.19', 30.12841796875),
  ('5223.7.1', 30.1279296875),
  ('5223.7.32', 30.126953125),
  ('5223.7.20', 30.12646484375),
  ('5223.7.37', 30.12646484375),
  ('5223.7.28', 30.12646484375),
  ('5223.7.2', 30.1259765625),
  ('5223.7.8', 30.1240234375),
  ('5223.7.14', 30.12353515625),
  ('3331.2', 30.12353515625),
  ('5223.7.15', 30.123046875),
  ('5223.7.29', 30.12158203125),
  ('5223.1', 30.11962890625),
  ('5223.7.24', 30.11962890625),
  ('5223.7.4', 30.119140625),
  ('5223.7.7', 30.1181640625),
  ('5223.7.27', 30.1171875),
  ('5223.7.30', 30.1171875),
  ('5223.7.11', 30.11669921875),
  ('3343.1.7', 30.1162109375),
  ('5223.7.13', 30.11572265625),
  ('5223.7.23', 30.115234375),
  ('5223.7.3

## Evaluation helpers

In [14]:
def evaluate_ranked_lists(cases, ranked_lists, label):
    rr, r5, r10 = [], [], []
    for case, ranked in zip(cases, ranked_lists):
        preds = [code for code, _ in ranked[:10]]
        target = case["target"]
        if target in preds:
            rr.append(1.0 / (preds.index(target) + 1))
        else:
            rr.append(0.0)
        r5.append(float(target in preds[:5]))
        r10.append(float(target in preds[:10]))
    return {
        "setting": label,
        "MRR": float(np.mean(rr)),
        "Recall@5": float(np.mean(r5)),
        "Recall@10": float(np.mean(r10)),
        "n_cases": len(cases),
    }


def minmax_scores(score_dict):
    if not score_dict:
        return {}
    values = np.array(list(score_dict.values()), dtype="float32")
    mn, mx = float(values.min()), float(values.max())
    denom = max(mx - mn, 1e-8)
    return {k: (float(v) - mn) / denom for k, v in score_dict.items()}


def hybrid_ranked_lists(cases, lateon_ranked, alpha):
    outputs = []
    for case, late_row in zip(cases, lateon_ranked):
        late_scores = {code: score for code, score in late_row[:LATEON_RETRIEVE_K]}
        markov_scores = top_markov_scores(case, MARKOV_CANDIDATE_K)
        union = set(late_scores) | set(markov_scores)

        late_norm = minmax_scores({c: late_scores.get(c, 0.0) for c in union})
        markov_norm = minmax_scores({c: markov_scores.get(c, 0.0) for c in union})

        combined = []
        for code in union:
            score = alpha * markov_norm.get(code, 0.0) + (1.0 - alpha) * late_norm.get(code, 0.0)
            combined.append((code, score))
        outputs.append(sorted(combined, key=lambda x: x[1], reverse=True))
    return outputs

In [15]:
results = []

for split, cases, lateon_ranked in [
    ("validation", validation_cases, validation_lateon),
    ("test", test_cases, test_lateon),
]:
    row = evaluate_ranked_lists(cases, lateon_ranked, "lateon_pure")
    row.update({"split": split, "alpha_markov": 0.0, "lateon_retrieve_k": LATEON_RETRIEVE_K})
    results.append(row)
    print(row)

    for alpha in ALPHAS:
        hybrid_ranked = hybrid_ranked_lists(cases, lateon_ranked, alpha)
        row = evaluate_ranked_lists(cases, hybrid_ranked, f"hybrid_alpha_{alpha:.2f}")
        row.update({"split": split, "alpha_markov": alpha, "lateon_retrieve_k": LATEON_RETRIEVE_K})
        results.append(row)
        print(row)

results_df = pd.DataFrame(results)
display(results_df.sort_values(["split", "MRR"], ascending=[True, False]).style.format({
    "MRR": "{:.4f}",
    "Recall@5": "{:.4f}",
    "Recall@10": "{:.4f}",
}))
results_df.to_csv("jobhop_v2_lighton_lateon_results.csv", index=False)
print("Saved: jobhop_v2_lighton_lateon_results.csv")

{'setting': 'lateon_pure', 'MRR': 0.06828492063492064, 'Recall@5': 0.1165, 'Recall@10': 0.158, 'n_cases': 2000, 'split': 'validation', 'alpha_markov': 0.0, 'lateon_retrieve_k': 200}
{'setting': 'hybrid_alpha_0.00', 'MRR': 0.06829126984126983, 'Recall@5': 0.1165, 'Recall@10': 0.158, 'n_cases': 2000, 'split': 'validation', 'alpha_markov': 0.0, 'lateon_retrieve_k': 200}
{'setting': 'hybrid_alpha_0.25', 'MRR': 0.12196130952380951, 'Recall@5': 0.199, 'Recall@10': 0.257, 'n_cases': 2000, 'split': 'validation', 'alpha_markov': 0.25, 'lateon_retrieve_k': 200}
{'setting': 'hybrid_alpha_0.50', 'MRR': 0.12109107142857142, 'Recall@5': 0.197, 'Recall@10': 0.2555, 'n_cases': 2000, 'split': 'validation', 'alpha_markov': 0.5, 'lateon_retrieve_k': 200}
{'setting': 'hybrid_alpha_0.75', 'MRR': 0.12524325396825398, 'Recall@5': 0.205, 'Recall@10': 0.282, 'n_cases': 2000, 'split': 'validation', 'alpha_markov': 0.75, 'lateon_retrieve_k': 200}
{'setting': 'hybrid_alpha_0.90', 'MRR': 0.12701825396825395, 'Reca

,setting,MRR,Recall@5,Recall@10,n_cases,split,alpha_markov,lateon_retrieve_k
11,hybrid_alpha_0.75,0.1445,0.2285,0.3105,2000,test,0.750000,200
12,hybrid_alpha_0.90,0.1431,0.2280,0.3120,2000,test,0.900000,200
13,hybrid_alpha_1.00,0.1387,0.2250,0.3085,2000,test,1.000000,200
9,hybrid_alpha_0.25,0.1330,0.2065,0.2740,2000,test,0.250000,200
10,hybrid_alpha_0.50,0.1327,0.2055,0.2730,2000,test,0.500000,200
8,hybrid_alpha_0.00,0.0763,0.1215,0.1680,2000,test,0.000000,200
7,lateon_pure,0.0762,0.1215,0.1680,2000,test,0.000000,200
5,hybrid_alpha_0.90,0.1270,0.2035,0.2830,2000,validation,0.900000,200
4,hybrid_alpha_0.75,0.1252,0.2050,0.2820,2000,validation,0.750000,200
6,hybrid_alpha_1.00,0.1244,0.1985,0.2765,2000,validation,1.000000,200


Saved: jobhop_v2_lighton_lateon_results.csv


In [16]:
reference_rows = [
    {
        "setting": "BGE-M3 hybrid, previous 09 sample",
        "split": "test",
        "alpha_markov": 0.50,
        "MRR": 0.1485,
        "Recall@5": 0.2320,
        "Recall@10": 0.3155,
        "note": "Sampled 2k test result from notebook 09.",
    },
    {
        "setting": "Qwen3 hybrid, previous 09 sample",
        "split": "test",
        "alpha_markov": 0.50,
        "MRR": 0.1479,
        "Recall@5": 0.2310,
        "Recall@10": 0.3200,
        "note": "Sampled 2k test result from notebook 09.",
    },
    {
        "setting": "MiniLM hybrid, previous 08 full test",
        "split": "test",
        "alpha_markov": 0.50,
        "MRR": 0.1343,
        "Recall@5": 0.2153,
        "Recall@10": 0.2926,
        "note": "Full test result from notebook 08.",
    },
    {
        "setting": "Pure Markov, previous 08 full test",
        "split": "test",
        "alpha_markov": 1.00,
        "MRR": 0.1312,
        "Recall@5": 0.2069,
        "Recall@10": 0.2919,
        "note": "Full test transition-count baseline.",
    },
]
comparison_df = pd.concat([results_df, pd.DataFrame(reference_rows)], ignore_index=True, sort=False)
display(comparison_df[["setting", "split", "alpha_markov", "MRR", "Recall@5", "Recall@10", "note"]].style.format({
    "alpha_markov": "{:.2f}",
    "MRR": "{:.4f}",
    "Recall@5": "{:.4f}",
    "Recall@10": "{:.4f}",
}))

,setting,split,alpha_markov,MRR,Recall@5,Recall@10,note
0,lateon_pure,validation,0.00,0.0683,0.1165,0.1580,nan
1,hybrid_alpha_0.00,validation,0.00,0.0683,0.1165,0.1580,nan
2,hybrid_alpha_0.25,validation,0.25,0.1220,0.1990,0.2570,nan
3,hybrid_alpha_0.50,validation,0.50,0.1211,0.1970,0.2555,nan
4,hybrid_alpha_0.75,validation,0.75,0.1252,0.2050,0.2820,nan
5,hybrid_alpha_0.90,validation,0.90,0.1270,0.2035,0.2830,nan
6,hybrid_alpha_1.00,validation,1.00,0.1244,0.1985,0.2765,nan
7,lateon_pure,test,0.00,0.0762,0.1215,0.1680,nan
8,hybrid_alpha_0.00,test,0.00,0.0763,0.1215,0.1680,nan
9,hybrid_alpha_0.25,test,0.25,0.1330,0.2065,0.2740,nan


## How to report this notebook

This is a LightOn LateOn screening run.

Report carefully:

- `lightonai/LateOn` is a late-interaction / ColBERT-style model, not a dense single-vector embedding model.
- The default result is sampled, not full test.
- Pure LateOn tests whether multi-vector semantic retrieval alone works for next-occupation prediction.
- Hybrid LateOn + Markov tests whether late-interaction semantics add complementary signal to transition counts.

If the parser fails because PyLate changes its output schema, inspect the first raw retrieval row printed by the notebook and adjust `parse_retrieved_row`.